# E2：多目标优化 —— 动力 vs 成本权衡（pymoo）
# E2: Multi-Objective Optimization — Performance vs Cost (pymoo)

## 一、实验原理 | Principle

### 1.1 为什么多目标？
真实工程很少只有一个目标。设计电动车：
- 想要**加速快** → 大功率电机
- 想要**便宜** → 小功率电机
两个目标矛盾，没有"最优解"，只有"权衡解"。

### 1.2 Pareto 前沿
**Pareto 最优**：无法在不牺牲一个目标的情况下改进另一个目标的解。
所有 Pareto 最优解的集合叫 **Pareto 前沿**。
工程师从前沿上按需求选：要性能选一头，要便宜选另一头，要均衡选中间。

### 1.3 NSGA-II
最经典的多目标遗传算法：
1. 随机生成一批设计（种群）
2. 按"支配关系"排序（谁在两个目标上都更好）
3. 保留前沿解，交叉变异产生下一代
4. 重复 50 代，前沿逐渐收敛

本实验用**真实 pymoo 库**跑 50 代进化，不是示意动画。

## 二、实验目的 | Objectives
1. 理解 Pareto 前沿的概念和工程意义
2. 掌握 pymoo 的 Problem 定义和 NSGA-II 调用
3. 从 Pareto 前沿中选择折中方案

## 三、学生任务 | Your Tasks
1. **运行**优化，观察 Pareto 前沿的形状（加速时间 vs 成本）
2. **选择**：如果预算 25 万，0-100km/h 能到几秒？如果要求 5 秒内，最便宜多少钱？
3. **改进**：加第三个目标"百公里电耗"，做三目标优化（提示：`n_obj=3`）
4. **思考题**：Pareto 前沿上的解，哪个是"最好"的？（提示：没有最好，只有最适合需求）

## 四、验证内容 | Verification
- [ ] pymoo 成功运行 50 代
- [ ] Pareto 解数量 ≥ 5
- [ ] 前沿图显示清晰的权衡曲线（非单点）

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| Pareto 只有 1-2 个解 | 两个目标无冲突（都想要大功率） | 检查目标函数是否有真实权衡 |
| `No module named 'pymoo'` | 内核选错 | 切换到 `Python 3.10 (FASTSim)` 内核（含 pymoo） |
| 优化很慢 | 种群/代数太大 | 减小 `pop_size` 或 `n_gen` |


In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np
print('✓ pymoo 导入成功')


In [ ]:
class EVDesign(Problem):
    """电动车设计：加速性能 vs 购车成本"""
    """权衡：大功率加速快但贵，小功率便宜但慢"""
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, n_constr=0,
                         xl=np.array([50., 1000.]), xu=np.array([200., 2500.]))
    def _evaluate(self, x, out, *args, **kwargs):
        power, weight = x[:, 0], x[:, 1]
        t_0100 = 2.5 * weight / power  # 加速时间
        cost = 15000 + 80 * power + 5 * weight  # 成本
        out['F'] = np.column_stack([t_0100, cost])

problem = EVDesign()
res = minimize(problem, NSGA2(pop_size=60), ('n_gen', 50),
               verbose=False, seed=42)
pareto = res.F
print(f'Pareto 解数: {len(pareto)}')
assert len(pareto) >= 5, 'Pareto 解太少!'
print('✅ 评测通过！')


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(pareto[:, 0], pareto[:, 1], c='blue', alpha=0.6)
plt.xlabel('0-100km/h Time (s)')
plt.ylabel('Cost (CNY)')
plt.title('Pareto Front: Performance vs Cost')
plt.grid(True, alpha=0.3)
plt.show()


## 🏆 挑战任务
1. 加第三个目标：百公里电耗，做三目标优化。
2. 从 Pareto 前沿选一个折中解，用 FASTSim 验证真实能耗。
